# Open-weight VLM roster — **remaining models only** (Kaggle, 2×T4)

Finishes the roster from the 2026-07-16 run. The three models that already completed
(`chexagent_8b`, `qwen25_vl_3b`, `qwen25_vl_7b`) are **not** re-run — their predictions
are already in that run's `outputs.zip`.

**This notebook runs (default):**

| group | transformers | models |
|---|---|---|
| A | `==4.40.0` | `chexagent_2_3b` — the specialist; hard-asserts this exact version |
| B | `>=4.50,<5` | `medgemma_4b_it` (gated → needs `HF_TOKEN`), `smolvlm2_2_2b` |
| C *(optional)* | `~=4.44` | `phi35_vision` — off by default; see `INCLUDE_PHI35` |

**Why one notebook needs three installs.** CheXagent-2-3b's remote code asserts
`transformers==4.40.0`; Gemma-3 / SmolVLM2 need `>=4.50`; Phi-3.5's remote code uses
`DynamicCache.seen_tokens`, removed after ~4.48. No single version satisfies all three, so
each group **reinstalls transformers and runs its models in a subprocess** — a fresh Python
process picks up whichever version is installed at launch. **The notebook kernel never
imports `transformers`**, so the reinstalls don't fight an already-loaded module (the exact
trap the env-guard warned about last time).

## Kaggle setup — REQUIRED
1. **Accelerator `GPU T4 x2`**, **Internet ON**.
2. Attach the crops dataset (`crops_v4a`, `manifest_v4a.json`, `crops_v4b`, `manifest_v4b.json`).
3. **`HF_TOKEN` secret** (Add-ons → Secrets) — only for gated `medgemma`; accept its license
   first. Without it, medgemma fail-skips and the rest still run.

## Fixes carried in from the last run
- `MAX_NEW_TOKENS = 120` — matches production `name_groq.py::BLIND_PROMPT` (was 64, not comparable).
- **dtype gates on Ampere (`capability[0] >= 8`)**, not `is_bf16_supported()` — the latter
  returns True on a T4 and silently selected emulated bf16.
- **A model that produces 0 usable rows is recorded FAILED**, never OK — last run's phi35 wrote
  400 empty rows and still reported OK, which would have scored as 0% hallucination.
- SmolVLM: `num2words` installed. Phi-3.5 (if enabled): a version where `seen_tokens` still exists.

## Output
`predictions_<model>_<dataset>.jsonl` (`{"id","kind","description"}`) — the schema
`score_naming.py` already reads. Scored locally, not here.

## 1. Config

In [ ]:
from pathlib import Path
import json, hashlib, os

# Match production so these numbers are comparable to the scout/pixtral/gemini runs.
MAX_NEW_TOKENS = 120        # name_groq.py uses max_tokens=120
INCLUDE_PHI35  = False      # general VLM; Qwen 3B/7B already cover the general axis + size axis

PROMPT = (
    "This is a small region that a diagnostic model focused on within a chest X-ray. "
    "Describe anything artificial, non-anatomical, or out of place that you see in this "
    "region. If everything looks like normal anatomy, say so. Be specific and concise."
)

INPUT_DIR = Path("/kaggle/input")
WORK_DIR  = Path("/kaggle/working")
OUT_DIR   = WORK_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"MAX_NEW_TOKENS={MAX_NEW_TOKENS} | INCLUDE_PHI35={INCLUDE_PHI35}")

## 2. Resolve crops (no `transformers` here)

Same recursive resolver + falsifying byte-hash control as the prior notebook: v4a/v4b share
filenames and row counts, so a mix-up is otherwise silent. Writes `datasets.json` for the
worker subprocesses to read.

In [ ]:
import zipfile
DATASET_TOKENS = ("v4a", "v4b")

def find_one(name):
    hits = sorted(INPUT_DIR.rglob(name));  return hits[0] if hits else None

def _claims_dataset(path, dataset):
    parts = str(path).lower().replace("\\", "/").split("/")
    for part in reversed(parts):
        names = [t for t in DATASET_TOKENS if t in part]
        if len(names) == 1:
            return names[0] == dataset
    return False

def resolve_crops_dir(dataset):
    cands = [d for d in sorted(INPUT_DIR.rglob("*"))
             if d.is_dir() and any(d.glob("*.png")) and _claims_dataset(d, dataset)]
    if len(cands) > 1:
        raise RuntimeError(f"Ambiguous crops for {dataset}: {cands} — refusing to guess.")
    if cands:
        return cands[0]
    z = find_one(f"crops_{dataset}.zip")
    if z is None:
        raise FileNotFoundError(f"No crops for {dataset} under /kaggle/input. Attach the dataset.")
    dest = WORK_DIR / f"crops_{dataset}"; dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(z) as zf: zf.extractall(dest)
    nested = dest / "crops"
    if not any(dest.glob("*.png")) and nested.is_dir() and any(nested.glob("*.png")):
        return nested
    return dest

def load_manifest(dataset):
    p = find_one(f"manifest_{dataset}.json")
    if p is None: raise FileNotFoundError(f"manifest_{dataset}.json not found under /kaggle/input")
    return json.load(open(p, encoding="utf-8"))

DATASETS = {}
for ds in ("v4a", "v4b"):
    cdir = resolve_crops_dir(ds); man = load_manifest(ds)
    missing = [r["id"] for r in man if not (cdir / f"{r['id']}.png").exists()]
    assert not missing, f"{ds}: {len(missing)} rows missing images, e.g. {missing[:5]}"
    DATASETS[ds] = {"crops_dir": str(cdir), "manifest_path": str(find_one(f'manifest_{ds}.json'))}
    n_pos = sum(1 for r in man if r["kind"] == "positive")
    print(f"{ds}: {len(man)} rows ({n_pos} pos, {len(man)-n_pos} ctrl) | crops -> {cdir}")

# falsifying control: prove v4a != v4b by bytes, not by path
def _digest(ds, n=20):
    h = hashlib.sha256()
    for p in sorted(Path(DATASETS[ds]["crops_dir"]).glob("*.png"))[:n]: h.update(p.read_bytes())
    return h.hexdigest()[:12]
_da, _db = _digest("v4a"), _digest("v4b")
assert _da != _db, f"v4a and v4b are byte-identical ({_da}) — same images resolved twice."
assert DATASETS["v4a"]["crops_dir"] != DATASETS["v4b"]["crops_dir"]
print(f"resolver control PASSED — distinct ({_da} vs {_db})")

(WORK_DIR / "datasets.json").write_text(json.dumps(DATASETS, indent=2))

## 3. HF auth (for gated `medgemma` only)

The token is exported as an **environment variable**, which every worker subprocess inherits
(`Popen` passes `os.environ` through) and which `huggingface_hub` reads automatically — so
gated downloads authenticate without depending on `huggingface_hub.login` being importable in
this kernel. `login()` is attempted too, but only best-effort.

In [ ]:
import os

def hf_login():
    # Getting the secret and setting the env var are the load-bearing steps — kept separate
    # so a failure tells you WHICH one broke (last run collapsed both into one ImportError).
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"No HF_TOKEN secret ({type(e).__name__}: {e}). "
              "medgemma will fail-skip; every other model is unaffected.")
        return False
    # Both names are honoured by huggingface_hub; subprocesses inherit these.
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    try:
        from huggingface_hub import login
        login(token=token)
    except Exception as e:
        print(f"  (huggingface_hub.login not usable here: {type(e).__name__}; "
              "env var is what the subprocess actually uses, so this is fine)")
    print("HF token set — gated models available to subprocesses")
    return True

HF_OK = hf_login()

## 4. Write the worker

One script, run once per group in its own process. It imports `transformers` fresh, so it
sees whichever version the group just installed. Per-model loader/generate conventions were
each verified against the live model card. **Empty output ⇒ FAILED**, so a model that loads
but generates nothing is never mistaken for a clean run.

In [ ]:
worker_src = r"""
import argparse, gc, json, time, traceback
from pathlib import Path
import torch
from PIL import Image

def _dtype():
    # bf16 only on real Ampere+ (compute capability major >= 8). T4 is 7.5 -> fp16.
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:
        return torch.bfloat16
    return torch.float16
DTYPE = _dtype()
N_GPUS = torch.cuda.device_count()

def bnb():
    from transformers import BitsAndBytesConfig
    return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=DTYPE,
                              bnb_4bit_quant_type="nf4")

# ---- chexagent_2_3b : AutoTokenizer + from_list_format (transformers==4.40.0) ----
def load_chexagent_2_3b(mid, q):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
    m = AutoModelForCausalLM.from_pretrained(mid, device_map='auto', trust_remote_code=True)
    return tok, m.to(DTYPE).eval()
def gen_chexagent_2_3b(tok, m, path, prompt, mnt):
    query = tok.from_list_format([{'image': path}, {'text': prompt}])
    conv = [{'from':'system','value':'You are a helpful assistant.'},{'from':'human','value':query}]
    ids = tok.apply_chat_template(conv, add_generation_prompt=True, return_tensors='pt')
    out = m.generate(ids.to(m.device), do_sample=False, num_beams=1, use_cache=True,
                     max_new_tokens=mnt)[0]
    return tok.decode(out[ids.size(1):-1]).strip()

# ---- generic ImageTextToText : medgemma, smolvlm2 ----
def load_generic_itt(mid, q):
    from transformers import AutoModelForImageTextToText, AutoProcessor
    proc = AutoProcessor.from_pretrained(mid, trust_remote_code=True)
    kw = dict(device_map='auto', trust_remote_code=True)
    if q:
        try:
            kw['quantization_config'] = bnb()
            return proc, AutoModelForImageTextToText.from_pretrained(mid, **kw)
        except Exception as e:
            print(f'  4-bit failed ({e!r}); fp16 fallback'); kw.pop('quantization_config', None)
    kw['torch_dtype'] = DTYPE
    return proc, AutoModelForImageTextToText.from_pretrained(mid, **kw)
def gen_generic_itt(proc, m, path, prompt, mnt):
    img = Image.open(path).convert('RGB')
    msg = [{'role':'user','content':[{'type':'image','image':img},{'type':'text','text':prompt}]}]
    inp = proc.apply_chat_template(msg, add_generation_prompt=True, tokenize=True,
                                   return_dict=True, return_tensors='pt').to(m.device)
    with torch.no_grad():
        out = m.generate(**inp, max_new_tokens=mnt, do_sample=False)
    new = out[:, inp['input_ids'].shape[1]:]
    return proc.batch_decode(new, skip_special_tokens=True)[0].strip()

# ---- phi35_vision : AutoModelForCausalLM + <|image_1|> ----
def load_phi35(mid, q):
    from transformers import AutoModelForCausalLM, AutoProcessor
    proc = AutoProcessor.from_pretrained(mid, trust_remote_code=True, num_crops=4)
    kw = dict(device_map='auto', trust_remote_code=True, _attn_implementation='eager')
    if q:
        try:
            kw['quantization_config'] = bnb()
            return proc, AutoModelForCausalLM.from_pretrained(mid, **kw)
        except Exception as e:
            print(f'  4-bit failed ({e!r}); fp16 fallback'); kw.pop('quantization_config', None)
    kw['torch_dtype'] = DTYPE
    return proc, AutoModelForCausalLM.from_pretrained(mid, **kw)
def gen_phi35(proc, m, path, prompt, mnt):
    img = Image.open(path).convert('RGB')
    msg = [{'role':'user','content':f'<|image_1|>\n{prompt}'}]
    text = proc.tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    inp = proc(text, [img], return_tensors='pt').to(m.device)
    with torch.no_grad():
        out = m.generate(**inp, max_new_tokens=mnt, do_sample=False,
                         eos_token_id=proc.tokenizer.eos_token_id)
    new = out[:, inp['input_ids'].shape[1]:]
    return proc.batch_decode(new, skip_special_tokens=True)[0].strip()

REG = {
  'chexagent_2_3b': (load_chexagent_2_3b, gen_chexagent_2_3b, 'StanfordAIMI/CheXagent-2-3b', False),
  'medgemma_4b_it': (load_generic_itt,   gen_generic_itt,     'google/medgemma-4b-it',       True),
  'smolvlm2_2_2b':  (load_generic_itt,   gen_generic_itt,     'HuggingFaceTB/SmolVLM2-2.2B-Instruct', True),
  'phi35_vision':   (load_phi35,         gen_phi35,           'microsoft/Phi-3.5-vision-instruct',    True),
}

def run_dataset(gen, tok, m, ds, manifest, cdir, out_path, prompt, mnt):
    n_ok = n_fail = 0; t0 = time.time()
    with open(out_path, 'w', encoding='utf-8') as f:
        for i, row in enumerate(manifest):
            try:
                desc = gen(tok, m, f"{cdir}/{row['id']}.png", prompt, mnt); n_ok += 1
            except Exception as e:
                desc = ''; n_fail += 1
                if n_fail <= 3: print(f'  IMAGE FAILED {row["id"]}: {e!r}')
            f.write(json.dumps({'id': row['id'], 'kind': row['kind'], 'description': desc}) + '\n')
            if (i+1) % 50 == 0: print(f'  {ds} {i+1}/{len(manifest)}')
    dt = time.time() - t0
    print(f'  {ds}: {n_ok}/{n_ok+n_fail} ok | {dt:.1f}s | {(n_ok+n_fail)/max(dt,1e-9):.2f} img/s')
    return {'dataset': ds, 'n_ok': n_ok, 'n_failed': n_fail, 'elapsed_s': round(dt, 1)}

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--models', nargs='+', required=True)
    ap.add_argument('--out-dir', required=True)
    ap.add_argument('--datasets-json', required=True)
    ap.add_argument('--prompt', required=True)
    ap.add_argument('--max-new-tokens', type=int, required=True)
    ap.add_argument('--group', required=True)
    a = ap.parse_args()
    import transformers
    print(f'worker up | transformers {transformers.__version__} | dtype {DTYPE} | {N_GPUS} GPU')
    DS = json.loads(Path(a.datasets_json).read_text())
    for ds in DS:  # attach parsed manifest
        DS[ds]['manifest'] = json.loads(Path(DS[ds]['manifest_path']).read_text())
    out = Path(a.out_dir); log = []
    for name in a.models:
        if name not in REG:
            print(f'SKIP unknown {name}'); continue
        load, gen, mid, q = REG[name]
        print(f"\n{'='*68}\n{name}  ({mid})\n{'='*68}")
        tok = m = None
        try:
            tok, m = load(mid, q); print('  loaded ok')
        except Exception as e:
            print(f'  FAILED to load — {type(e).__name__}: {e}'); traceback.print_exc()
            log.append({'model': name, 'model_id': mid, 'status': 'FAILED_LOAD',
                        'reason': f'{type(e).__name__}: {e}'})
            del tok, m; gc.collect(); torch.cuda.empty_cache(); continue
        results = []
        try:
            for ds, d in DS.items():
                op = out / f'predictions_{name}_{ds}.jsonl'
                results.append(run_dataset(gen, tok, m, ds, d['manifest'], d['crops_dir'],
                                           op, a.prompt, a.max_new_tokens))
            total_ok = sum(r['n_ok'] for r in results)
            # Empty output => FAILED, not OK (fixes last run's phi35 false-OK).
            status = 'OK' if total_ok > 0 else 'FAILED_INFERENCE'
            reason = None if total_ok > 0 else 'loaded but produced 0 usable rows across all datasets'
            log.append({'model': name, 'model_id': mid, 'status': status,
                        'reason': reason, 'results': results})
        except Exception as e:
            print(f'  FAILED during inference — {type(e).__name__}: {e}'); traceback.print_exc()
            log.append({'model': name, 'model_id': mid, 'status': 'FAILED_INFERENCE',
                        'reason': f'{type(e).__name__}: {e}', 'results': results})
        finally:
            del tok, m; gc.collect(); torch.cuda.empty_cache()
    (out / f'run_log_{a.group}.json').write_text(json.dumps(log, indent=2))
    print(f'\nwrote run_log_{a.group}.json')

if __name__ == '__main__':
    main()
"""
Path("/kaggle/working/worker.py").write_text(worker_src)
print("worker.py written")

## 5. Group runner

In [ ]:
import subprocess, sys, json
DS_JSON = str(WORK_DIR / "datasets.json")

def run_group(group, models, pip_spec):
    print(f"\n########## GROUP {group}: pip install {pip_spec} ##########")
    # -q keeps logs short; reinstalls transformers for THIS group only.
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pip_spec.split()])
    if r.returncode != 0:
        print(f"pip install failed for group {group}; skipping"); return
    cmd = [sys.executable, "-u", "/kaggle/working/worker.py",
           "--models", *models, "--out-dir", str(OUT_DIR), "--datasets-json", DS_JSON,
           "--prompt", PROMPT, "--max-new-tokens", str(MAX_NEW_TOKENS), "--group", group]
    # stream child output live
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1)
    for line in p.stdout: print(line, end="")
    p.wait()
    print(f"group {group} exit code: {p.returncode}")

## 6. Group A — CheXagent-2-3b (`transformers==4.40.0`)

In [ ]:
run_group("A", ["chexagent_2_3b"], "transformers==4.40.0 accelerate einops")

## 7. Group B — MedGemma + SmolVLM2 (`transformers>=4.50,<5`)

In [ ]:
run_group("B", ["medgemma_4b_it", "smolvlm2_2_2b"],
          "transformers>=4.50,<5 accelerate bitsandbytes num2words")

## 8. Group C — Phi-3.5-vision *(optional)*

Off by default. Phi-3.5's remote code needs `DynamicCache.seen_tokens`, removed after ~4.48,
so it wants its own `~=4.44` install. Enable via `INCLUDE_PHI35 = True` in cell 1.

In [ ]:
if INCLUDE_PHI35:
    run_group("C", ["phi35_vision"],
              "transformers==4.44.2 accelerate bitsandbytes")
else:
    print("Phi-3.5 skipped (INCLUDE_PHI35=False).")

## 9. Summary + zip

In [ ]:
import json, shutil
log = []
for g in ("A", "B", "C"):
    p = OUT_DIR / f"run_log_{g}.json"
    if p.exists(): log.extend(json.loads(p.read_text()))

n_ok = sum(1 for e in log if e["status"] == "OK")
print(f"{n_ok}/{len(log)} models produced predictions\n")
lines = ["# VLM roster — remaining models — run log\n\n",
         f"- MAX_NEW_TOKENS={MAX_NEW_TOKENS}\n- models OK: {n_ok}/{len(log)}\n\n"]
for e in log:
    print(f"{e['model']:16s} {e['status']}")
    lines.append(f"## {e['model']} (`{e['model_id']}`)\n\nStatus: **{e['status']}**\n\n")
    if e["status"] == "OK":
        for r in e["results"]:
            print(f"     {r['dataset']}: {r['n_ok']}/{r['n_ok']+r['n_failed']} ok")
            lines.append(f"- {r['dataset']}: {r['n_ok']}/{r['n_ok']+r['n_failed']} ok, {r['elapsed_s']}s\n")
    else:
        print(f"     reason: {str(e.get('reason'))[:120]}")
        lines.append(f"- reason: `{e.get('reason')}`\n")
    lines.append("\n")

(OUT_DIR / "run_log_remaining.md").write_text("".join(lines), encoding="utf-8")
shutil.make_archive(str(WORK_DIR / "outputs_remaining"), "zip", OUT_DIR)
print(f"\nwrote {WORK_DIR/'outputs_remaining.zip'} — download via the Output tab")